# Module 8 Notebook: ML Plan

Before building a model, define the prediction task clearly. A model is only as useful as the question you ask it to learn.

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0Aimport+pandas+as+pd%0A%0Atry%3A%0A++++from+sklearn.model_selection+import+train_test_split%0A++++sklearn_available+%3D+True%0Aexcept+ModuleNotFoundError%3A%0A++++sklearn_available+%3D+False%0A++++print%28%27scikit-learn+is+not+available+here%3B+using+a+small+classroom+train%2Ftest+split+fallback.%27%29%0A%0A++++def+train_test_split%28X%2C+y%2C+test_size%3D0.25%2C+random_state%3D42%2C+stratify%3DNone%29%3A%0A++++++++data+%3D+X.copy%28%29%0A++++++++data%5B%27_target%27%5D+%3D+y.values%0A++++++++if+stratify+is+not+None%3A%0A++++++++++++test_parts+%3D+%5B%5D%0A++++++++++++train_parts+%3D+%5B%5D%0A++++++++++++for+_%2C+group+in+data.groupby%28%27_target%27%2C+group_keys%3DFalse%29%3A%0A++++++++++++++++shuffled+%3D+group.sample%28frac%3D1%2C+random_state%3Drandom_state%29%0A++++++++++++++++n_test+%3D+max%281%2C+round%28len%28shuffled%29+%2A+test_size%29%29%0A++++++++++++++++test_parts.append%28shuffled.iloc%5B%3An_test%5D%29%0A++++++++++++++++train_parts.append%28shuffled.iloc%5Bn_test%3A%5D%29%0A++++++++++++test+%3D+pd.concat%28test_parts%29.sample%28frac%3D1%2C+random_state%3Drandom_state%29%0A++++++++++++train+%3D+pd.concat%28train_parts%29.sample%28frac%3D1%2C+random_state%3Drandom_state%29%0A++++++++else%3A%0A++++++++++++shuffled+%3D+data.sample%28frac%3D1%2C+random_state%3Drandom_state%29%0A++++++++++++n_test+%3D+round%28len%28shuffled%29+%2A+test_size%29%0A++++++++++++test+%3D+shuffled.iloc%5B%3An_test%5D%0A++++++++++++train+%3D+shuffled.iloc%5Bn_test%3A%5D%0A++++++++return+train.drop%28columns%3D%27_target%27%29%2C+test.drop%28columns%3D%27_target%27%29%2C+train%5B%27_target%27%5D%2C+test%5B%27_target%27%5D%0A%0Adf+%3D+pd.read_csv%28%27phase2_study_support_clean.csv%27%29%0Adf.head%28%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
import pandas as pd

try:
    from sklearn.model_selection import train_test_split
    sklearn_available = True
except ModuleNotFoundError:
    sklearn_available = False
    print('scikit-learn is not available here; using a small classroom train/test split fallback.')

    def train_test_split(X, y, test_size=0.25, random_state=42, stratify=None):
        data = X.copy()
        data['_target'] = y.values
        if stratify is not None:
            test_parts = []
            train_parts = []
            for _, group in data.groupby('_target', group_keys=False):
                shuffled = group.sample(frac=1, random_state=random_state)
                n_test = max(1, round(len(shuffled) * test_size))
                test_parts.append(shuffled.iloc[:n_test])
                train_parts.append(shuffled.iloc[n_test:])
            test = pd.concat(test_parts).sample(frac=1, random_state=random_state)
            train = pd.concat(train_parts).sample(frac=1, random_state=random_state)
        else:
            shuffled = data.sample(frac=1, random_state=random_state)
            n_test = round(len(shuffled) * test_size)
            test = shuffled.iloc[:n_test]
            train = shuffled.iloc[n_test:]
        return train.drop(columns='_target'), test.drop(columns='_target'), train['_target'], test['_target']

df = pd.read_csv('phase2_study_support_clean.csv')
df.head()

## 1. Define the target

Our target is `needs_support`. The model will try to predict whether a fictional learner might need extra support.

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0Atarget+%3D+%27needs_support%27%0Afeatures+%3D+%5B%27practice_quiz_avg%27%2C%27weekly_study_hours%27%2C%27sleep_hours%27%2C%27missing_assignments%27%2C%27screen_time_hours%27%2C%27attends_help_session%27%5D%0A%0AX+%3D+df%5Bfeatures%5D.copy%28%29%0Ay+%3D+df%5Btarget%5D.copy%28%29%0AX.head%28%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
target = 'needs_support'
features = ['practice_quiz_avg','weekly_study_hours','sleep_hours','missing_assignments','screen_time_hours','attends_help_session']

X = df[features].copy()
y = df[target].copy()
X.head()

## 1A. Choose the kind of algorithm carefully

An algorithm is the learning method. You do not need to master every algorithm yet, but you should know that different algorithms learn in different ways.

- **Decision tree:** asks split-style questions, like a flowchart.
- **Logistic regression:** finds weighted patterns for yes/no classification.
- **k-nearest neighbors:** compares a new case to similar past cases.

For this beginner project, a decision tree is a good first choice because it is easy to explain.

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0Aalgorithm_choice+%3D+%7B%0A++++%27task%27%3A+%27classification%27%2C%0A++++%27starter_algorithm%27%3A+%27DecisionTreeClassifier%27%2C%0A++++%27why_this_one%27%3A+%27It+is+beginner-friendly+and+easier+to+explain+than+many+black-box+models.%27%0A%7D%0Aalgorithm_choice"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
algorithm_choice = {
    'task': 'classification',
    'starter_algorithm': 'DecisionTreeClassifier',
    'why_this_one': 'It is beginner-friendly and easier to explain than many black-box models.'
}
algorithm_choice

## 2. Convert categories into numbers

Most starter ML models need numeric input.

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0AX%5B%27attends_help_session%27%5D+%3D+X%5B%27attends_help_session%27%5D.map%28%7B%27yes%27%3A+1%2C+%27no%27%3A+0%7D%29%0Ay+%3D+y.map%28%7B%27yes%27%3A+1%2C+%27no%27%3A+0%7D%29%0AX.head%28%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
X['attends_help_session'] = X['attends_help_session'].map({'yes': 1, 'no': 0})
y = y.map({'yes': 1, 'no': 0})
X.head()

## 3. Create train and test sets

The model learns from training data. It is judged on testing data it did not train on.

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0AX_train%2C+X_test%2C+y_train%2C+y_test+%3D+train_test_split%28%0A++++X%2C+y%2C+test_size%3D0.25%2C+random_state%3D42%2C+stratify%3Dy%0A%29%0A%0Aprint%28%27Training+rows%3A%27%2C+len%28X_train%29%29%0Aprint%28%27Testing+rows%3A%27%2C+len%28X_test%29%29%0Aprint%28%27Target+balance+in+full+data%3A%27%29%0Aprint%28y.value_counts%28normalize%3DTrue%29.round%282%29%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))
print('Target balance in full data:')
print(y.value_counts(normalize=True).round(2))

## 3A. Where validation fits

Professional teams often use three sets: train, validation, and test. The validation set helps tune choices before the final test. To keep this first project manageable, we will use train and test only.

The habit still matters: do not keep changing the model after peeking at the final test score.

## 4. Set a simple baseline

A model should beat a simple guess. If most examples are `no`, always guessing `no` is the baseline to beat.

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0Abaseline_guess+%3D+y_train.mode%28%29%5B0%5D%0Abaseline_accuracy+%3D+%28y_test+%3D%3D+baseline_guess%29.mean%28%29%0Aprint%28%27Baseline+guess%3A%27%2C+baseline_guess%29%0Aprint%28%27Baseline+accuracy%3A%27%2C+round%28baseline_accuracy%2C+3%29%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
baseline_guess = y_train.mode()[0]
baseline_accuracy = (y_test == baseline_guess).mean()
print('Baseline guess:', baseline_guess)
print('Baseline accuracy:', round(baseline_accuracy, 3))

## 5. Risk check

This is a fictional learning-support example. In the real world, a model like this should never be used to label students without human review, context, and privacy protections.

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0Aplan+%3D+%7B%0A++++%27target%27%3A+target%2C%0A++++%27features%27%3A+features%2C%0A++++%27model_type%27%3A+%27classification%27%2C%0A++++%27baseline_to_beat%27%3A+round%28float%28baseline_accuracy%29%2C+3%29%2C%0A++++%27human_review_needed%27%3A+True%0A%7D%0Aplan"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
plan = {
    'target': target,
    'features': features,
    'model_type': 'classification',
    'baseline_to_beat': round(float(baseline_accuracy), 3),
    'human_review_needed': True
}
plan

## Portfolio note

Save your target, features, train/test split, baseline, and one responsible-use warning.

## 6. Transfer the same planning move to other domains

The notebook uses a synthetic support dataset, but the planning logic also applies to delivery, email, playlists, customer renewal, and sports examples.

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0Adomain_transfer+%3D+pd.DataFrame%28%5B%0A++++%7B%27domain%27%3A+%27Email%27%2C+%27target%27%3A+%27spam_or_not%27%2C+%27possible_features%27%3A+%27sender+domain%2C+subject+words%2C+link+count%27%2C+%27one_limit%27%3A+%27new+scams+may+use+words+the+model+has+never+seen%27%7D%2C%0A++++%7B%27domain%27%3A+%27Delivery%27%2C+%27target%27%3A+%27late_or_on_time%27%2C+%27possible_features%27%3A+%27distance%2C+weather%2C+pickup+time%2C+traffic+level%27%2C+%27one_limit%27%3A+%27holiday+traffic+may+not+match+normal+weeks%27%7D%2C%0A++++%7B%27domain%27%3A+%27Playlist%27%2C+%27target%27%3A+%27skip_or_replay%27%2C+%27possible_features%27%3A+%27tempo%2C+genre%2C+artist+familiarity%2C+previous+skips%27%2C+%27one_limit%27%3A+%27taste+changes+over+time%27%7D%2C%0A++++%7B%27domain%27%3A+%27Customer+renewal%27%2C+%27target%27%3A+%27renew_or_cancel%27%2C+%27possible_features%27%3A+%27usage%2C+support+tickets%2C+plan+age%27%2C+%27one_limit%27%3A+%27one+season+of+data+may+not+generalize%27%7D%2C%0A%5D%29%0Adomain_transfer"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
domain_transfer = pd.DataFrame([
    {'domain': 'Email', 'target': 'spam_or_not', 'possible_features': 'sender domain, subject words, link count', 'one_limit': 'new scams may use words the model has never seen'},
    {'domain': 'Delivery', 'target': 'late_or_on_time', 'possible_features': 'distance, weather, pickup time, traffic level', 'one_limit': 'holiday traffic may not match normal weeks'},
    {'domain': 'Playlist', 'target': 'skip_or_replay', 'possible_features': 'tempo, genre, artist familiarity, previous skips', 'one_limit': 'taste changes over time'},
    {'domain': 'Customer renewal', 'target': 'renew_or_cancel', 'possible_features': 'usage, support tickets, plan age', 'one_limit': 'one season of data may not generalize'},
])
domain_transfer

## 7. Feature timing and leakage check

A feature is only fair game if it would be available before the prediction and does not already reveal the answer.

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0Afeature_review+%3D+pd.DataFrame%28%7B%0A++++%27feature%27%3A+features+%2B+%5B%27project_score%27%2C+%27student_code%27%5D%2C%0A++++%27available_before_prediction%27%3A+%5BTrue%2C+True%2C+True%2C+True%2C+True%2C+True%2C+False%2C+True%5D%2C%0A++++%27leaks_answer_or_identifier%27%3A+%5BFalse%2C+False%2C+False%2C+False%2C+False%2C+False%2C+True%2C+True%5D%2C%0A++++%27use_in_first_model%27%3A+%5BTrue%2C+True%2C+True%2C+True%2C+True%2C+True%2C+False%2C+False%5D%2C%0A%7D%29%0Afeature_review"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
feature_review = pd.DataFrame({
    'feature': features + ['project_score', 'student_code'],
    'available_before_prediction': [True, True, True, True, True, True, False, True],
    'leaks_answer_or_identifier': [False, False, False, False, False, False, True, True],
    'use_in_first_model': [True, True, True, True, True, True, False, False],
})
feature_review

## 8. Write a model plan card

Before training, summarize what the model is and is not allowed to claim.

<div style="display:flex; align-items:center; justify-content:space-between; border-left:4px solid #999; border-radius:6px; background:rgba(153,153,153,0.08); padding:8px 20px 8px 16px; margin:6px 0;">
  <span>&#128269;&nbsp;<b>Stuck on this code?</b></span>
  <span style="display:flex; align-items:center; gap:8px;">
    <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0Amodel_plan_card+%3D+%7B%0A++++%27prediction_question%27%3A+%27Can+the+model+predict+the+synthetic+needs_support+label+from+available+practice+signals%3F%27%2C%0A++++%27target%27%3A+target%2C%0A++++%27features%27%3A+features%2C%0A++++%27split%27%3A+%2775%25+train+%2F+25%25+test+with+stratified+labels%27%2C%0A++++%27baseline%27%3A+f%22always+predict+%7Bbaseline_guess%7D%22%2C%0A++++%27success_condition%27%3A+%27model+should+beat+the+baseline+and+have+explainable+mistakes%27%2C%0A++++%27not_allowed_claim%27%3A+%27This+does+not+identify+real+learners+or+make+real+support+decisions.%27%2C%0A%7D%0Amodel_plan_card"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat" alt="Explain with ChatGPT" style="height:26px; vertical-align:middle;"></a>
  </span>
</div>

In [ ]:
model_plan_card = {
    'prediction_question': 'Can the model predict the synthetic needs_support label from available practice signals?',
    'target': target,
    'features': features,
    'split': '75% train / 25% test with stratified labels',
    'baseline': f"always predict {baseline_guess}",
    'success_condition': 'model should beat the baseline and have explainable mistakes',
    'not_allowed_claim': 'This does not identify real learners or make real support decisions.',
}
model_plan_card